<img src=https://courseware.decoded.com/images/decoded/logo-decoded.png align=left width=100px/>

# GitHub, version control and reproducible environments

---
<small>(c) 2026 Decoded Limited. All rights reserved. Website: https://decoded.com/</small>


## Programmes

* Data Science Accelerator - Data science project - GitHub

## Content

This notebook shows how to package earlier analysis work into a small, shareable Python project.

You will connect the EDA and preprocessing workflow to professional project habits: Git, GitHub, dependency management, virtual environments and a simple saved SQL query.


## How to use this completed reference notebook

Use this version after the code-along to check your work, revisit the workflow, or recover if your local notebook gets out of sync.

### Introductions

#### Purpose: What is the aim of this content?

In EDA, you practised exploring university performance data. You moved from EDA into preprocessing for modelling.

In this workshop you will practise turning that kind of analysis into a small project that another person can understand, run and safely build on.

You will use version control to track changes, record dependencies so the environment is reproducible, and save a small SQL query as part of the project.


#### Objectives: What will I learn?

By the end of this notebook, you will be able to:

* connect EDA and preprocessing work to a simple project handover
* explain what Git and GitHub are used for
* create a local Git repository and make meaningful commits
* use a branch to work on a change safely
* describe a typical GitHub collaboration workflow
* record Python dependencies using `requirements.txt`
* explain why virtual environments make Python projects more reliable
* save a simple SQL query as a version-controlled project file
* create a small project structure that is easier to share and maintain


#### Process: How will I learn this content?

You will work through:

* short explanations
* runnable Python demonstrations
* command-line examples to discuss or try locally
* small challenges with completed reference code
* reflection prompts to connect the workflow to your own work


#### Applications: Why does this matter?

EDA and preprocessing are most useful when someone else can understand how the work was done and repeat it.

Version control and environment management help you answer practical questions:

* What changed since the last version?
* Why did it change?
* Can someone else run this project?
* Have we recorded the files needed to recreate the work?
* Can we review the analysis logic without opening every notebook cell manually?


This notebook is not a new analytics technique notebook. It is about making earlier analytics work easier to manage.

From Module 3, we carry forward:

* loading CSV files into pandas
* checking rows, columns, data types and missing values
* cleaning column names and simple text values
* grouping and summarising data
* communicating useful findings
* joining lookup data
* choosing a target column
* removing leakage before modelling
* separating features from the target
* preparing data for modelling

From now on, we hope that you are beginning to think more end to end: how analysis, preprocessing, modelling decisions and outputs fit together in a project someone else may need to review.

This section adds the project habits that support all of this work: Git history, GitHub collaboration, dependency files, virtual environments and a simple folder structure.


---


## The Scenario: A University Analytics Project

You are continuing the university performance work from M3 and M4.

The team now wants a small project that:

* stores Python code for an analysis report
* records the libraries used in EDA and preprocessing
* keeps a useful SQL summary query in a project file
* saves generated outputs in a separate folder
* uses Git to track each meaningful change

We will build a lightweight version of that workflow.


### Import libraries

In [1]:
# Starter imports
# These libraries help us work with folders, files, Git commands and a small database.

from pathlib import Path
import shutil
import sqlite3
import subprocess
import sys
import textwrap

# Create a project folder path.
# TODO: keep the folder name short and descriptive.
project_dir = Path("week_5_university_project_demo").resolve()

project_dir

PosixPath('/Users/james.lam/Workshops/Data Science Accelerator/Data science projects/week_5_university_project_demo')

### Create the directory and remove old project direction if it exists

In [2]:
# Create a clean project folder

if project_dir.exists():
    shutil.rmtree(project_dir)

(project_dir / "sql").mkdir(parents=True)
(project_dir / "reports").mkdir()
(project_dir / "src").mkdir()

In [3]:
# Check the project folder structure

print(f"Project folder created at: {project_dir}")
print("Folders:")

for path in sorted(project_dir.iterdir()):
    print("-", path.name)

Project folder created at: /Users/james.lam/Workshops/Data Science Accelerator/Data science projects/week_5_university_project_demo
Folders:
- reports
- sql
- src


The folder structure separates different kinds of work:

* `src/` stores Python code
* `sql/` stores saved SQL queries
* `reports/` stores generated outputs

This mirrors the end-to-end learning journey: analysis logic, supporting query logic and outputs are kept in clear places.


---


## Part 1: What Git tracks

Git is a version control tool. It stores a history of changes to files in a project.

A useful commit should answer two questions:

* What changed?
* Why is this change useful?

Git does not replace clear thinking. It gives your clear thinking a history.


In [4]:
# Helper function: run Git commands from this notebook
# You do not need to memorise this function.
# Its job is to send Git commands to the project folder and print the result.

def run_git(args, cwd=project_dir):
    result = subprocess.run(
        ["git", *args],
        cwd=cwd,
        text=True,
        capture_output=True,
        check=True,
    )

    output = result.stdout.strip() or result.stderr.strip()
    if output:
        print(output)

    return result

#### Run Git commands to initialise the repo

In [5]:
# Initialise Git

# initialise the repository on a branch called main.
run_git(["init", "-b", "main"])

# set a local Git username and email for this practice repository.
run_git(["config", "user.name", "lab-learner"])
run_git(["config", "user.email", "your.email@example.com"])

# check the Git status.
run_git(["status", "--short"])

Initialized empty Git repository in /Users/james.lam/Workshops/Data Science Accelerator/Data science projects/week_5_university_project_demo/.git/


CompletedProcess(args=['git', 'status', '--short'], returncode=0, stdout='', stderr='')

`git status --short` gives a compact view of files that are untracked, changed or staged.

At this point, Git knows the project folder exists, but it is not yet tracking any project files.


---


## Part 2: Make the first commit

A commit is a saved checkpoint in the project history.

We will start with a `README.md` file. A good README helps someone understand what the project is for and how to run it.


### Create a Readme file 

In [6]:
# Create a README file
# A README explains what the project is and how someone should use it.

readme = project_dir / "README.md"

readme.write_text(textwrap.dedent('''
# University Analytics Project Demo

This project packages earlier EDA and preprocessing work into a small Python project.

## Project structure

- `src/`: Python code
- `sql/`: saved SQL query
- `reports/`: generated outputs

## How to use this project

1. Create a virtual environment.
2. Install the dependencies.
3. Run the report script.
''').strip() + "\n")

print(readme.read_text())

# University Analytics Project Demo

This project packages earlier EDA and preprocessing work into a small Python project.

## Project structure

- `src/`: Python code
- `sql/`: saved SQL query
- `reports/`: generated outputs

## How to use this project

1. Create a virtual environment.
2. Install the dependencies.
3. Run the report script.



### Run the Git commands to add the Readme file

In [7]:
# First commit
# The pattern is: check status -> add file -> commit change -> check log.

run_git(["status", "--short"])
run_git(["add", "README.md"])
run_git(["commit", "-m", "Add project README"])
run_git(["log", "--oneline", "--max-count", "3"])

?? README.md
[main (root-commit) ab45d39] Add project README
 1 file changed, 15 insertions(+)
 create mode 100644 README.md
ab45d39 Add project README


CompletedProcess(args=['git', 'log', '--oneline', '--max-count', '3'], returncode=0, stdout='ab45d39 Add project README\n', stderr='')

A simple Git workflow is:

1. Make a focused change.
2. Check the status.
3. Stage the files you want to include.
4. Commit with a clear message.
5. Repeat.


---


## Part 3: Use a branch for a change

A branch lets you work on a change without immediately changing the main project line.

Teams often use branches for features, fixes and experiments.


### Guidance

In this section we deliberately make a very small code change first. The fuller end-to-end report comes later, once the database and SQL query exist.

#### Create a new branch

In [8]:
# Create a branch for a small change

# create and switch to a new branch.
run_git(["switch", "-c", "add-starter-report-script"])

# create a small starter Python script.
report_script = project_dir / "src" / "summarise_university_outcomes.py"

report_script.write_text("print('University outcome report coming soon')\n")

print(report_script.read_text())

Switched to a new branch 'add-starter-report-script'
print('University outcome report coming soon')



#### Discuss

Pause and explain
* In your own words, what did we add to the project?

* Write a short answer as a comment below.
* Example: We added a starter Python script inside the src folder.

* We added a starter Python script inside the src folder.

#### Check the status of the Git

In [9]:
# Commit the branch change

run_git(["status", "--short"])
run_git(["add", "src/summarise_university_outcomes.py"])
run_git(["commit", "-m", "Add starter report script"])
run_git(["log", "--oneline", "--max-count", "3"])

?? src/
[add-starter-report-script 08e664e] Add starter report script
 1 file changed, 1 insertion(+)
 create mode 100644 src/summarise_university_outcomes.py
08e664e Add starter report script
ab45d39 Add project README


CompletedProcess(args=['git', 'log', '--oneline', '--max-count', '3'], returncode=0, stdout='08e664e Add starter report script\nab45d39 Add project README\n', stderr='')

A branch is most useful when the change has a clear purpose.

The branch name `add-university-report` describes the work. The commit message describes the saved change.


---


## Part 4: GitHub collaboration workflow

Git runs locally on your machine. GitHub hosts a copy of the repository online so people can collaborate.

A common workflow is:

1. Create or clone a repository.
2. Create a branch for your work.
3. Commit local changes.
4. Push the branch to GitHub.
5. Open a pull request.
6. Discuss, review and test the change.
7. Merge the change when it is ready.

Example commands, to discuss or try in your own terminal:

```bash
git remote add origin https://github.com/your-org/your-repo.git
git push -u origin add-university-report
```

A pull request is a structured conversation about a proposed change. It is where the team reviews the code, asks questions and checks that the change is ready.


In [10]:
# Merge the branch back into main

run_git(["branch", "--show-current"])
run_git(["switch", "main"])
run_git(["merge", "add-starter-report-script"])
run_git(["log", "--oneline", "--max-count", "5"])

add-starter-report-script
Switched to branch 'main'
Updating ab45d39..08e664e
Fast-forward
 src/summarise_university_outcomes.py | 1 +
 1 file changed, 1 insertion(+)
 create mode 100644 src/summarise_university_outcomes.py
08e664e Add starter report script
ab45d39 Add project README


CompletedProcess(args=['git', 'log', '--oneline', '--max-count', '5'], returncode=0, stdout='08e664e Add starter report script\nab45d39 Add project README\n', stderr='')

---


## Part 5: Dependencies and virtual environments

A dependency is code your project needs but does not define itself. For example, many data projects depend on packages such as `pandas`, `numpy` or `requests`.

A virtual environment is an isolated Python environment for one project. It helps prevent one project's package versions from interfering with another project.

Typical setup commands:

```bash
python -m venv .venv
source .venv/bin/activate      # macOS or Linux
.venv\Scripts\Activate.ps1     # Windows PowerShell
python -m pip install -r requirements.txt
```

The runnable demo mostly uses Python's standard library, but the dependency file records the libraries used across the wider M3 and M4 workflow: pandas, NumPy, Matplotlib and scikit-learn.


In [11]:
# Record project dependencies
# These are libraries used in earlier EDA and preprocessing work.

requirements = project_dir / "requirements.txt"

requirements.write_text(textwrap.dedent('''
pandas
numpy
matplotlib
scikit-learn
''').strip() + "\n")

print(requirements.read_text())

pandas
numpy
matplotlib
scikit-learn



In [12]:
# Record Python version and files Git should ignore

python_version = project_dir / ".python-version"
python_version.write_text(f"{sys.version_info.major}.{sys.version_info.minor}\n")

gitignore = project_dir / ".gitignore"
gitignore.write_text(textwrap.dedent('''
# Python cache files
__pycache__/
*.py[cod]

# Local virtual environments
.venv/
decoded_env/

# Local databases and generated reports
*.db
reports/*.csv
''').strip() + "\n")

for file_name in ["requirements.txt", ".python-version", ".gitignore"]:
    print(f"\n--- {file_name} ---")
    print((project_dir / file_name).read_text())


--- requirements.txt ---
pandas
numpy
matplotlib
scikit-learn


--- .python-version ---
3.11


--- .gitignore ---
# Python cache files
__pycache__/
*.py[cod]

# Local virtual environments
.venv/
decoded_env/

# Local databases and generated reports
*.db
reports/*.csv



In [13]:
# Commit the environment setup files

run_git(["add", "requirements.txt", ".python-version", ".gitignore"])
run_git(["commit", "-m", "Record Python environment setup"])
run_git(["status", "--short"])

[main 7d9a91c] Record Python environment setup
 3 files changed, 16 insertions(+)
 create mode 100644 .gitignore
 create mode 100644 .python-version
 create mode 100644 requirements.txt


CompletedProcess(args=['git', 'status', '--short'], returncode=0, stdout='', stderr='')

`requirements.txt` records the packages needed to run the project.

`.gitignore` tells Git which local files should not be committed. This usually includes virtual environments, temporary files, local databases and generated outputs.


---


## Part 6: Keep SQL simple and version-controlled

Some analytics projects include a small amount of SQL. At this stage, you do not need a full database migration workflow.

A simple beginner-friendly habit is:

* keep important SQL in a `.sql` file
* give the file a clear name
* commit it with the Python code that uses it
* avoid committing local database files such as `.db`

We will create a tiny practice database based on the M3 university scenario, then save one SQL query as a project file.


In [14]:
# Create a small practice database
# This keeps SQL light. We only need a tiny table so we can save and run one query.

db_path = project_dir / "university_performance.db"

with sqlite3.connect(db_path) as conn:
    conn.executescript('''
    DROP TABLE IF EXISTS universities;

    CREATE TABLE universities (
        university_name TEXT,
        region TEXT,
        satisfaction_score REAL,
        strong_outcome INTEGER
    );

    INSERT INTO universities VALUES
        ('Northbridge University', 'North', 84.5, 1),
        ('Greenfield College', 'South', 77.2, 0),
        ('Riverside Institute', 'North', 81.0, 1),
        ('Lakeside University', 'West', 72.4, 0),
        ('City Central University', 'South', 88.1, 1);
    ''')

print(f"Practice database created: {db_path.name}")

Practice database created: university_performance.db


In [15]:
# Save one SQL query as a project file

query_path = project_dir / "sql" / "regional_outcome_summary.sql"

query_path.write_text(textwrap.dedent('''
SELECT
    region,
    COUNT(*) AS universities,
    ROUND(AVG(satisfaction_score), 1) AS average_satisfaction,
    SUM(strong_outcome) AS strong_outcomes
FROM universities
GROUP BY region
ORDER BY average_satisfaction DESC;
''').strip() + "\n")

print(query_path.read_text())

SELECT
    region,
    COUNT(*) AS universities,
    ROUND(AVG(satisfaction_score), 1) AS average_satisfaction,
    SUM(strong_outcome) AS strong_outcomes
FROM universities
GROUP BY region
ORDER BY average_satisfaction DESC;



In [16]:
# Run the saved SQL query

with sqlite3.connect(project_dir / "university_performance.db") as conn:
    rows = conn.execute(query_path.read_text()).fetchall()

for row in rows:
    print(row)

('North', 2, 82.8, 2)
('South', 2, 82.7, 1)
('West', 1, 72.4, 0)


In [17]:
# Commit the saved SQL query

run_git(["add", "sql/regional_outcome_summary.sql"])
run_git(["commit", "-m", "Add regional outcome summary SQL query"])
run_git(["status", "--short"])

[main f4ab66a] Add regional outcome summary SQL query
 1 file changed, 8 insertions(+)
 create mode 100644 sql/regional_outcome_summary.sql


CompletedProcess(args=['git', 'status', '--short'], returncode=0, stdout='', stderr='')

The database file itself is ignored because it is a generated local file.

The SQL query is tracked because it explains how the report is created. This is the important beginner habit: track the useful source files, not local outputs.


---


## Part 7: Run the project end to end 

Now we can run the Python report against the small practice database and saved SQL query.


In [ ]:
# Upgrade the report script so it runs end to end
# This is the larger piece of code, so read it in sections:
# 1. file paths
# 2. read SQL query
# 3. run query
# 4. save CSV report

# The script will create a CSV report in the `reports/` folder.
report_script = project_dir / "src" / "summarise_university_outcomes.py"


Created regional_outcome_summary.csv with 3 rows

region,universities,average_satisfaction,strong_outcomes
North,2,82.8,2
South,2,82.7,1
West,1,72.4,0



In [ ]:
# this is to create the report script that will run the SQL query and save the results to a CSV file.
report_script.write_text(textwrap.dedent('''
from pathlib import Path
import csv
import sqlite3

PROJECT_DIR = Path(__file__).resolve().parents[1]
DB_PATH = PROJECT_DIR / "university_performance.db"
QUERY_PATH = PROJECT_DIR / "sql" / "regional_outcome_summary.sql"
REPORT_PATH = PROJECT_DIR / "reports" / "regional_outcome_summary.csv"

query = QUERY_PATH.read_text()

with sqlite3.connect(DB_PATH) as conn:
    rows = conn.execute(query).fetchall()

REPORT_PATH.parent.mkdir(exist_ok=True)

with REPORT_PATH.open("w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["region", "universities", "average_satisfaction", "strong_outcomes"])
    writer.writerows(rows)

print(f"Created {REPORT_PATH.name} with {len(rows)} rows")
''').strip() + "\n")


In [ ]:
# Run the report script and check the output
result = subprocess.run(
    [sys.executable, str(report_script)],
    cwd=project_dir,
    text=True,
    capture_output=True,
    check=True,
)

print(result.stdout)
print((project_dir / "reports" / "regional_outcome_summary.csv").read_text())

In [19]:
# Check Git status after generating the report
# The CSV report is ignored by Git because it is generated output.

run_git(["status", "--short"])

M src/summarise_university_outcomes.py


CompletedProcess(args=['git', 'status', '--short'], returncode=0, stdout=' M src/summarise_university_outcomes.py\n', stderr='')

The report output is useful, but it is generated. In many projects, generated outputs are not committed unless the team has a clear reason to keep them in the repository.

The source files that create the output are the durable part of the project.


---


## Challenge: Prepare a project for sharing

Use the project you created in this notebook as a reference.

For your own Python project, prepare these files:

* `README.md` with the project purpose and setup steps
* `.gitignore` with local files excluded
* `requirements.txt` with dependencies recorded
* a `sql/` folder for any saved SQL queries
* at least three meaningful commits

Success criteria:

* another person can understand the project from the README
* Git history shows clear, focused changes
* generated local files are not committed by accident
* useful SQL queries are stored as reviewable files
* dependency setup is recorded clearly


### Optional Stretch

Add a second SQL file called `high_satisfaction_universities.sql`.

Write a query that returns universities with a `satisfaction_score` above 80. Commit the new SQL file with a clear message.


---


## Reflection

Use these questions to check your understanding:

* What files should be committed in a small Python analytics project?
* What files should usually be ignored?
* How does a branch reduce risk when changing a project?
* What does a virtual environment protect you from?
* Why is a saved SQL query easier to review than an unexplained database file?


## Key Takeaways

* Git records a meaningful history of project changes.
* GitHub helps teams review, discuss and merge changes.
* Branches let you work safely before merging into the main project.
* Virtual environments keep project dependencies isolated.
* `requirements.txt` records the packages needed to recreate an environment.
* Saved SQL files make important queries visible, reviewable and easier to reuse.
* A professional project should help the next person understand how to run and change it.
